# Machine Learning Mini Project
## Topic 8: SMS Spam Message Detection Using Naive Bayes and Support Vector Machines (SVM)
**Course:** Introduction to Machine Learning (IML)  
**Institution:** Unitedworld Institute of Technology (UIT), Karnavati University, Gandhinagar, India  
**Author:** Ayush Panchal  
**Enrollment / KUID:** KU2507U0431  
**Semester / Program:** B.Tech Computer Science & Engineering (Cloud Computing)  
**Dataset Source:** UCI Machine Learning Repository / Kaggle SMS Spam Collection Dataset  

---
### Project Objective
The primary objective of this project is to develop an automated Natural Language Processing (NLP) and Machine Learning pipeline capable of accurately classifying incoming Short Message Service (SMS) text messages into **Legitimate (Ham)** or **Spam (Unsolicited/Fraudulent)**. We evaluate and compare probabilistic models (**Multinomial Naive Bayes**, **Complement Naive Bayes**, **Bernoulli Naive Bayes**) against structural margin classifiers (**Linear SVM**, **Kernel RBF SVM**) and a linear baseline (**Logistic Regression**).


In [ ]:
import os
import re
import string
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import joblib

from nltk.stem import PorterStemmer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.naive_bayes import MultinomialNB, BernoulliNB, ComplementNB
from sklearn.svm import LinearSVC, SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc, roc_auc_score
)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline
print("Libraries imported successfully.")


## 1. Dataset Acquisition and Exploration
The SMS Spam Collection is a public set of 5,574 SMS tagged messages that have been collected for SMS Spam research. It contains one set of SMS messages in English of 5,574 messages, tagged acording to being ham (legitimate) or spam.


In [ ]:
df = pd.read_csv('dataset.csv')
print(f"Total samples: {len(df)}")
df.head(5)


In [ ]:
# Class distribution and summary statistics
print(df['label'].value_counts())
print("\nPercentage breakdown:")
print(df['label'].value_counts(normalize=True) * 100)


## 2. Text Preprocessing and Tokenization
Textual data requires systematic normalization:
1. **Case Normalization:** Conversion of all characters to lowercase.
2. **Entity Masking:** Replacing raw URLs with `httpaddr`, email addresses with `emailaddr`, monetary symbols with `moneysymb`, phone numbers with `phonenum`, and generic digits with `numbr`.
3. **Punctuation Removal:** Eliminating special characters.
4. **Stopword Filtering:** Removing common uninformative English words.
5. **Stemming:** Applying the Porter Stemming algorithm to reduce inflectional forms to word stems.


In [ ]:
stemmer = PorterStemmer()
stopwords_set = set(ENGLISH_STOP_WORDS)

def clean_sms(text):
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', ' httpaddr ', text)
    text = re.sub(r'\S+@\S+', ' emailaddr ', text)
    text = re.sub(r'[£$€₹]', ' moneysymb ', text)
    text = re.sub(r'\b\d{5,}\b', ' phonenum ', text)
    text = re.sub(r'\b\d+\b', ' numbr ', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = text.split()
    cleaned = [stemmer.stem(w) for w in tokens if w not in stopwords_set and len(w) > 1]
    return " ".join(cleaned)

df['clean_message'] = df['message'].apply(clean_sms)
df[['message', 'clean_message']].head(5)


## 3. Train-Test Splitting & TF-IDF Vectorization
We split the dataset using an **80:20** stratified ratio to preserve the 86.6% ham to 13.4% spam ratio across both splits.

The mathematical formulation for Term Frequency-Inverse Document Frequency (TF-IDF) is:
$$\text{TF-IDF}(t, d, D) = \text{TF}(t, d) \times \log \left( \frac{1 + |D|}{1 + |\{d \in D : t \in d\}|} \right) + 1$$
We employ unigrams and bigrams $(1, 2)$ with sublinear term-frequency scaling.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_message'], df['label_num'], test_size=0.20, random_state=42, stratify=df['label_num']
)

tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=4000, sublinear_tf=True)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(f"X_train shape: {X_train_tfidf.shape}")
print(f"X_test shape : {X_test_tfidf.shape}")


## 4. Model Training & Mathematical Foundations
### A. Multinomial Naive Bayes
Based on Bayes' theorem under conditional independence:
$$P(C_k \mid \mathbf{x}) \propto P(C_k) \prod_{i=1}^{n} P(x_i \mid C_k)$$
With Laplace smoothing ($\alpha = 0.2$):
$$\hat{\theta}_{ki} = \frac{N_{ki} + \alpha}{N_k + \alpha d}$$

### B. Support Vector Machines (SVM)
Constructs an optimal separating hyperplane maximizing the geometric margin:
$$\min_{\mathbf{w}, b, \xi} \frac{1}{2} \|\mathbf{w}\|^2 + C \sum_{i=1}^{m} \xi_i \quad \text{subject to} \quad y_i (\mathbf{w}^T \phi(\mathbf{x}_i) + b) \ge 1 - \xi_i, \quad \xi_i \ge 0$$


In [ ]:
models = {
    'Multinomial Naive Bayes': MultinomialNB(alpha=0.2),
    'Complement Naive Bayes': ComplementNB(alpha=0.2),
    'Bernoulli Naive Bayes': BernoulliNB(alpha=0.2),
    'Linear SVM (LinearSVC)': LinearSVC(C=1.0, random_state=42, max_iter=2000),
    'Support Vector Machine (RBF)': SVC(kernel='rbf', C=1.0, random_state=42),
    'Logistic Regression': LogisticRegression(C=1.0, max_iter=1000, random_state=42)
}

results = []
for name, clf in models.items():
    clf.fit(X_train_tfidf, y_train)
    y_pred = clf.predict(X_test_tfidf)
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    results.append({
        'Model': name,
        'Accuracy (%)': round(acc * 100, 2),
        'Precision (%)': round(prec * 100, 2),
        'Recall (%)': round(rec * 100, 2),
        'F1-Score (%)': round(f1 * 100, 2)
    })

results_df = pd.DataFrame(results).set_index('Model')
results_df


## 5. Comparative Evaluation Visualizations
Let's inspect the confusion matrices, ROC curves, and performance metric charts.


In [ ]:
from IPython.display import Image, display
display(Image(filename='results/04_confusion_matrices.png'))
display(Image(filename='results/05_model_comparison_bar.png'))
display(Image(filename='results/06_roc_curves.png'))


## 6. Real-Time Inference Demo
Test custom SMS messages with confidence probabilities:


In [ ]:
nb_clf = models['Multinomial Naive Bayes']
svm_clf = models['Linear SVM (LinearSVC)']

def predict_custom(msg):
    cleaned = clean_sms(msg)
    vec = tfidf.transform([cleaned])
    nb_pred = 'SPAM' if nb_clf.predict(vec)[0] == 1 else 'HAM'
    nb_prob = nb_clf.predict_proba(vec)[0]
    svm_pred = 'SPAM' if svm_clf.predict(vec)[0] == 1 else 'HAM'
    svm_dec = svm_clf.decision_function(vec)[0]
    
    print(f"Message: {msg}")
    print(f"-> Naive Bayes: {nb_pred} (Ham: {nb_prob[0]*100:.1f}%, Spam: {nb_prob[1]*100:.1f}%)")
    print(f"-> Linear SVM : {svm_pred} (Decision Score: {svm_dec:+.3f})\n")

predict_custom("URGENT! You have won £1,000 cash prize. Call 08712400602 to claim now.")
predict_custom("Hey Ayush, are you coming to the college library for machine learning study?")